In [ ]:
from pyspark.sql import SparkSession

MONGO_URI = "mongodb://127.0.0.1:27017"
MONGO_DATABASE = "spark_demo"
MONGO_COLLECTION = "people"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PySparkMongoDBDemo")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .getOrCreate()
)

spark.version

:: loading settings :: url = jar:file:/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/suanki/.ivy2.5.2/cache
The jars for the packages stored in: /home/suanki/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c335389f-bedd-41d9-bfeb-97c1a1f2ef23;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;11.1.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
:: resolution report :: resolve 1803ms :: artifacts dl 22ms
	:: modules in use:
	org.mongodb#bson;5.1.4 from central in [default]
	org.mongodb#bson-record-codec;5.1.4 from central in

'4.2.0'

## Write a DataFrame

This overwrites only the demo `spark_demo.people` collection each time the cell runs.

In [2]:
people = spark.createDataFrame(
    [(1, "Ada", 36), (2, "Grace", 42), (3, "Linus", 35)],
    ["id", "name", "age"],
)

(
    people.write
    .format("mongodb")
    .mode("overwrite")
    .option("database", MONGO_DATABASE)
    .option("collection", MONGO_COLLECTION)
    .save()
)

## Read the collection back into Spark

In [3]:
people_from_mongo = (
    spark.read
    .format("mongodb")
    .option("database", MONGO_DATABASE)
    .option("collection", MONGO_COLLECTION)
    .load()
)

people_from_mongo.show(truncate=False)

people_from_mongo.printSchema()

+------------------------+---+---+-----+
|_id                     |age|id |name |
+------------------------+---+---+-----+
|6af69afa4fff79048ce8be0a|36 |1  |Ada  |
|6af69afa4fff79048ce8be09|42 |2  |Grace|
|6af69afa4fff79048ce8be0b|35 |3  |Linus|
+------------------------+---+---+-----+

root
 |-- _id: string (nullable = true)
 |-- age: long (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [4]:
## db["people"].find()  to show all the data


In [ ]:
## reading database table in parallel
numPartition: 

(
dbDataFrame = spark.read.format("jdbc")
.option("url", url).option("dbtable", tablename).option("driver", driver)
.option("numPartitions", 10).load()
)

In [5]:
spark.stop()